In [1]:
# !pip install -U transformers accelerate bitsandbytes peft trl datasets huggingface_hub
# !pip install "transformers>=4.41,<5.0" "trl<1.0" "bitsandbytes<0.49" "peft<0.19" "accelerate<1.10" --force-reinstall
# !pip install --no-deps "transformers==4.57.6" "trl==0.29.1" "bitsandbytes==0.48.2" "peft==0.18.1" "accelerate==1.9.0"
# import torch, torchvision
# print(torch.__version__, torchvision.__version__)
# !pip freeze > requirements_exp6.txt
# !pip show transformers accelerate peft trl bitsandbytes 2>/dev/null | grep -E "Name|Version"
!pip install --no-deps peft==0.18.1 trl==0.29.1 bitsandbytes==0.48.2
import torch, torchvision  # sanity check immediately after — should still both import cleanly

In [6]:
import transformers, trl, accelerate
print(transformers.__version__)
print(trl.__version__)
print(accelerate.__version__)


from huggingface_hub import login
from datasets import load_dataset
from datasets import Dataset
from huggingface_hub import login, HfApi
import pandas as pd


import os
os.environ["HF_TOKEN"] = "hf_token" # removed for safety reasons

from huggingface_hub import login
login(os.environ["HF_TOKEN"])

dataset_path = "businessrules/dataset_stratified_test"

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import (
    prepare_model_for_kbit_training,
    get_peft_model,
    LoraConfig,
)

4.57.1
0.29.1
1.11.0


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
2026-08-11 06:59:46.639136: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1786431586.833795      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786431586.887404      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786431587.354372      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786431587.354415      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoi

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B"
# MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-base"
# MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B"


OUTPUT_MODEL = "businessrules/Qwen-base-br-qlora_exp17" 

dataset = load_dataset(dataset_path, split="train")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
import torch

# 0 Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"    

# MAX_LENGTH = 2048
MAX_LENGTH = 4096

def format_example(example):
    return f"""### Instruction:
You are given a code snippet.
Extract the business rules implemented by the code.
Return ONLY the business rules in Markdown format.

### Code:
{example["cd"]}

### Business Rules:
{example["br"]}
"""

# 

def tokenize(example):
    # 1. Use your exact format_example logic
    full_text = format_example(example) + tokenizer.eos_token
    
    # This must match format_example EXACTLY up to the start of the answer
    prompt_text = f"""### Instruction:
You are given a code snippet.
Extract the business rules implemented by the code.
Return ONLY the business rules in Markdown format.

### Code:
{example["cd"]}

### Business Rules:
"""

    # 2. Tokenize with space for manual EOS
    full_tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_tensors=None 
    )

    # 3. Ensure EOS is present (Repetition Fix)
    if full_tokens["input_ids"][-1] != tokenizer.eos_token_id:
        full_tokens["input_ids"][-1] = tokenizer.eos_token_id

    prompt_tokens = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_tensors=None
    )

    # 4. Create Labels (Masking Fix)
    input_ids = full_tokens["input_ids"]
    labels = [ -100 if i < len(prompt_tokens["input_ids"]) else input_ids[i] for i in range(len(input_ids))]
    
    # Ensure the very last token (EOS) is NOT masked
    labels[-1] = tokenizer.eos_token_id
    
    full_tokens["labels"] = labels
    return full_tokens

tokenized_dataset = dataset.map(
    tokenize,
    remove_columns=dataset.column_names
)


# 1️ Load model in 8-bit
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.float16,
    device_map="auto"
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    trust_remote_code=True,
)

model.config.use_cache = False

# 2️ Prepare model for k-bit training (optional but recommended)
model = prepare_model_for_kbit_training(model)

# 3️ Enable gradient checkpointing
model.gradient_checkpointing_enable()

# 4️ Apply LoRA
lora_config = LoraConfig(
    r=20,
    lora_alpha=16,
    lora_dropout=0.11,
    target_modules=["q_proj", "v_proj"], #for Qwen
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# Print trainable params
model.print_trainable_parameters()

# 5️ Training arguments (8-bit friendly)
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=5e-4,
    num_train_epochs=8,
    fp16=False,      # turn off mixed precision
    bf16=False,      # explicitly disable BF16
    logging_steps=10,
    save_steps=500,
    save_total_limit=2,
    optim="paged_adamw_8bit",  # 8-bit optimizer
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    report_to="none"
)

from transformers import DataCollatorForSeq2Seq

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,        # Adding model here helps the collator handle internal padding requirements
    padding=True,
    pad_to_multiple_of=8
)


# 6️ Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    data_collator=collator,
    args=training_args
)

# print("World size:", trainer.args.world_size)

trainer.train()

trainer.model.save_pretrained(OUTPUT_MODEL)
tokenizer.save_pretrained(OUTPUT_MODEL)
  
trainer.model.push_to_hub(OUTPUT_MODEL)
tokenizer.push_to_hub(OUTPUT_MODEL)